**Complete LlamaIndex + HuggingFace RAG Pipeline**

- Document Loading
- HuggingFace Embedding Model
- Local HuggingFace LLM (Mistral)
- Prompt Configuration
- Chunking configuration
- Vector index creation
- Query engine
- Asking questions

|Parameter|Mistral-7B|TinyLlama|Reason|
|--|--|--|--|
|`model_name`|`mistralai/Mistral-7B-Instruct-v0.1`|`TinyLlama/TinyLlama-1.1B-Chat-v1.0`|Smaller model|
|`context_window`|4096|2048|TinyLlama is trained with a 2K context window|
|`torch_dtype`|`float16`|`float32` (CPU)|Better compatibility on CPU|
|`max_new_tokens`|512|256|Faster generation|

**Memory Requirements**

|Model|Parameters|RAM Needed (CPU)|Suitable for 16 GB RAM?|
|--|--|--|--|
|TinyLlama-1.1B|1.1 Billion|~4–6 GB|✅ Yes|
|Phi-3 Mini|3.8 Billion|~8–10 GB|✅ Yes|
|Mistral-7B|7 Billion|~16–20+ GB|⚠️ Borderline/Slow|
|Llama-3.1-8B|8 Billion|~18–22+ GB|❌ No|

**Performance Comparison**

|Feature|TinyLlama|Mistral-7B|
|--|--|--|
|Model Size|1.1B|7B|
|Download Size|~2 GB|~14 GB|
|CPU Speed|Fast|Slow|
|Response Quality|Good|Excellent|
|Learning RAG|⭐⭐⭐⭐⭐|⭐⭐⭐⭐|
|Production|Small applications|Enterprise applications|

- Base Model → Knows language but doesn't necessarily know how to follow your requests.

- Instruct Model → Trained to understand and follow instructions like a helpful assistant.

**Analogy**
Imagine two new employees.

**Employee A (Base Model)**
You ask:"Summarize this document."

**Employee A says:** "This document contains information about..."

Sometimes they may summarize, sometimes they may continue writing, or even generate unrelated text because they were only trained to predict the next word.

**Employee B (Instruct Model)**

You ask: "Summarize this document in three bullet points."

**Employee B responds:**

- Point 1
- Point 2
- Point 3

They follow your instructions because they were specifically trained to do so.

**Install Required Packages**

In [ ]:
!pip install llama-index
!pip install llama-index-llms-huggingface
!pip install llama-index-embeddings-huggingface

!pip install transformers
!pip install accelerate
!pip install sentence-transformers
!pip install torch

  Using cached llama_index_llms_huggingface-0.7.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.4 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 83.0.0
    Uninstalling setuptools-83.0.0:
      Successfully uninstalled setuptools-83.0.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1

**Import Required Libraries**

In [ ]:
import torch
from llama_index.core import(
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    PromptTemplate
)

from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

**Define Model Prompt**

For Mistral-Instruct models, use Mistral chat format.

link - https://github.com/run-llama/llama_index/blob/main/llama-index-integrations/llms/llama-index-llms-huggingface/llama_index/llms/huggingface/base.py

**Mistral**

In [ ]:
system_prompt = """
You are an intelligent Q&A assistant.

Your task it:
- Answer questions accurately.
- Use only the provided context.
- If the answer is not available in the context, say:
  "I don't know based on the provided documents."
"""

query_wrapper_prompt = PromptTemplate(
    "<s>[INST] {query_str} [/INST]"
)

**TinyLlama**

In [ ]:
system_prompt = """
You are a helpful AI assistant.
Answer questions only using the provided context.
If the answer is not found in the context, say:
"I don't know based on the provided documents."
"""

query_wrapper_prompt = PromptTemplate(
    "<|user|>\n{query_str}\n<|assistant|>\n"
)

**Configure HuggingFace LLM**

**Mistral**

In [ ]:
llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=512,
    generate_kwargs={
        "temperature":0.2,
        "do_sample":False
    },
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    model_kwargs={"torch_dtype":torch.float16}
)

**TinyLlama**

In [ ]:
llm = HuggingFaceLLM(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    context_window=2048,
    max_new_tokens=256,
    generate_kwargs={
        "temperature": 0.2,
        "do_sample": False
    },
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.float32
    }
)

**Configure Embedding Model**

Using: sentence-transformers/all-mpnet-base-v2

In [ ]:
embed_model = HuggingFaceEmbedding(

    model_name=
    "sentence-transformers/all-mpnet-base-v2"
)

**Global Settings**

In [ ]:
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 1024
Settings.chunk_overlap = 200

**Load Documents**

In [ ]:
documents = SimpleDirectoryReader("/content/data/").load_data()

**Create Vector Index**

In [ ]:
print(f"Number of documents: {len(documents)}")

Number of documents: 3


In [ ]:
index = VectorStoreIndex.from_documents(
    documents
)

**Create Query Engine**

In [ ]:
query_engine = index.as_query_engine(
    similarity_top_k=2
)

**Query Your RAG System**

In [ ]:
response = query_engine.query(
    "What is a self attention"
)
print(response)

The original query is as follows: What is a self attention
We have provided an existing answer: A self attention is a mechanism used in neural networks to attend to specific parts of an input sequence. It is a type of attention mechanism that involves computing the attention score for each input token based on its position in the sequence. This attention score is then used to weight the output of the network, allowing it to focus on the most relevant information in the input sequence. In the context of deep learning, self attention is often used in sequence modeling tasks, such as language modeling and natural language processing.
We have the opportunity to refine the existing answer (only if needed) with some more context below.

/content/data/Deep Learning.pdf

'81?%En%p+ܱ5n3uBN0!ct }zzxjXfO;"㭍-AJ=po& &rIrބ:E>Ng"qב,Êz#
JȅK>w|LUMy}MيP7Hi;!2v3c
